In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, save_npz


DATA_DIR = Path(r"../datasets")
ROOT_DIR = Path(r"../")
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = ROOT_DIR / "model_artifacts"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model directory:", MODEL_DIR)


Model directory: ..\model_artifacts


In [4]:
required_files = [
    PROCESSED_DIR / "weighted_matrix.npz",
    PROCESSED_DIR / "phase2_movies.csv",
    PROCESSED_DIR / "cf_user_ids.npy",
    PROCESSED_DIR / "cf_movie_ids.npy",
    PROCESSED_DIR / "cf_user_factors.npy",
    PROCESSED_DIR / "cf_movie_factors.npy",
]

for path in required_files:
    print(path.name, "->", path.exists())


weighted_matrix.npz -> True
phase2_movies.csv -> True
cf_user_ids.npy -> True
cf_movie_ids.npy -> True
cf_user_factors.npy -> True
cf_movie_factors.npy -> True


In [5]:
## 2. Copy/prepare artifacts

import shutil

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required artifact: {path}"
        )

for path in required_files:
    shutil.copy2(
        path,
        MODEL_DIR / path.name
    )

print("Artifacts copied.")


Artifacts copied.


In [6]:
# Use the best values from Phase 7 after evaluation.
config = {
    "positive_rating_threshold": 4.0,
    "content_weight": 0.5,
    "collaborative_weight": 0.5,
    "default_recommendation_count": 10,
    "model_version": "phase8-v1"
}

with open(
    MODEL_DIR / "model_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


{
  "positive_rating_threshold": 4.0,
  "content_weight": 0.5,
  "collaborative_weight": 0.5,
  "default_recommendation_count": 10,
  "model_version": "phase8-v1"
}


In [7]:
## Create a metadata summary

weighted_matrix = load_npz(
    MODEL_DIR / "weighted_matrix.npz"
)

movies = pd.read_csv(
    MODEL_DIR / "phase2_movies.csv"
)

user_ids = np.load(
    MODEL_DIR / "cf_user_ids.npy"
)

movie_ids = np.load(
    MODEL_DIR / "cf_movie_ids.npy"
)

user_factors = np.load(
    MODEL_DIR / "cf_user_factors.npy"
)

movie_factors = np.load(
    MODEL_DIR / "cf_movie_factors.npy"
)

metadata = {
    "movie_count": int(len(movies)),
    "content_feature_count": int(weighted_matrix.shape[1]),
    "content_matrix_rows": int(weighted_matrix.shape[0]),
    "collaborative_user_count": int(len(user_ids)),
    "collaborative_movie_count": int(len(movie_ids)),
    "user_factor_dimensions": list(user_factors.shape),
    "movie_factor_dimensions": list(movie_factors.shape),
    "model_version": config["model_version"]
}

with open(
    MODEL_DIR / "model_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

display(pd.DataFrame([metadata]))


,movie_count,content_feature_count,content_matrix_rows,collaborative_user_count,collaborative_movie_count,user_factor_dimensions,movie_factor_dimensions,model_version
0,62423,20185,62423,162541,56559,"[162541, 50]","[56559, 50]",phase8-v1


In [8]:
assert weighted_matrix.shape[0] == len(movies)
assert user_factors.shape[0] == len(user_ids)
assert movie_factors.shape[0] == len(movie_ids)
assert config["content_weight"] + config["collaborative_weight"] == 1.0

print("=" * 60)
print("PHASE 8 MODEL PACKAGING PASSED")
print("=" * 60)

for file in sorted(MODEL_DIR.iterdir()):
    print(file.name)


PHASE 8 MODEL PACKAGING PASSED
cf_movie_factors.npy
cf_movie_ids.npy
cf_user_factors.npy
cf_user_ids.npy
model_config.json
model_metadata.json
phase2_movies.csv
weighted_matrix.npz
